In [44]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [45]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
data = loader.load()

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

all_splits = text_splitter.split_documents(data)

In [47]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'mps'},
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5763.63it/s]


In [48]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(all_splits)

['52145aad-ec29-4d82-8648-f367777a30d3',
 'f9b133ac-2afe-4a0a-972e-3ac9307f925f',
 '3661426d-fcf4-47a3-b20b-4d56ee709239',
 '97b9d261-6581-442e-bf18-0b6824a473df',
 '011a49fc-0170-492e-bde7-7ae746cebbdc',
 '1a4be104-531b-413d-8ca8-d3ad85861b26',
 '5bf758f1-0bec-4757-8030-5156d47fc83e',
 'bee67a62-7296-41fb-bd3f-82773978c287',
 '3efe5443-a962-49f2-9bfa-37f75606f243',
 '04fd70d4-eb16-4e6f-8b8e-fe4882d5720f',
 '09835e59-0da8-4074-a135-a1c5064b2b25',
 'ee702c7c-06f7-415e-9fe4-06e4d178bf36',
 '55bdfe24-ae39-4d5a-8b30-0edf3ee5d375',
 '05e24c91-4155-4b20-bdb7-ccde09435799',
 'ac43ae81-d1f8-42ff-9463-522705e1a5f5',
 '04d71bae-bbdf-45c9-9d95-f7eae94d8b15',
 'a8b4b47c-81b8-4971-96fb-0f7e066f776d',
 '1122faad-859c-49ce-95ce-e8efb8cd9cf8',
 '5b1d6aaa-3b48-4abd-b6a1-591ee6d32520',
 '0eede2e7-4c13-4811-bac2-1f37e947bcf5',
 'ff35d86b-cfaf-4077-92d1-499eb5adc209',
 'b5492d85-b984-4bb7-b29b-7735ff6c7130',
 '695e8b88-20af-4c71-a10a-e3ab8cceacce',
 '172481eb-6822-45f7-8c40-30d05613c0b2',
 'd59a2ec7-59be-

In [66]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [67]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""

Answer the question based on the following context. If no relevant information is found in the context,

please say "I cannot find the relevant information from the provided context".

Context: {context}

Question: {question}

Answer: """)

In [68]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model=os.getenv("OLLAMA_MODEL"))

In [69]:
chain = (
{
# Searcher input: question string, output: Document list
# Lambda function input: Document list, output: merged text string
"context": retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
# RunnablePassthrough input: question string, output: pass the question string as is
"question": RunnablePassthrough()
}
# prompt input: dictionary {"context": text, "question": question}, output: formatted prompt template string
| prompt
# llm input: prompt template string, output: ChatMessage object
|llm
# StrOutputParser input: ChatMessage object, output: answer text string
| StrOutputParser()
)

In [72]:
question = "What is artificial intelligence?"

In [75]:
retriever_output = retriever.invoke(question)
print("Retriever output:", retriever_output, "\n")

Retriever output: [Document(id='3efe5443-a962-49f2-9bfa-37f75606f243', metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}, page_content='Glossary\nGlossary\nvte\nArtificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to gener

In [78]:
context = "\n\n".join(doc.page_content for doc in retriever_output)
print("Context passed to LLM:", context, "\n")

Context passed to LLM: Glossary
Glossary
vte
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

McCarthy defines intelligence as "the computational part of the ability to achieve goals in the world".[403] Another AI founder, Marvin Minsky, similarly descri

In [81]:
prompt_output = prompt.invoke({"context": context, "question": question})
print("Prompt output:", prompt_output, "\n")

Prompt output: messages=[HumanMessage(content='\n\nAnswer the question based on the following context. If no relevant information is found in the context,\n\nplease say "I cannot find the relevant information from the provided context".\n\nContext: Glossary\nGlossary\nvte\nArtificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widel

In [84]:
llm_output = llm.invoke(prompt_output)
print("LLM output:", llm_output, "\n")

LLM output: content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.' additional_kwargs={} response_metadata={'model': 'gemma4:26b', 'created_at': '2026-06-07T09:18:23.905896Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21644833208, 'load_duration': 13228162792, 'prompt_eval_count': 547, 'prompt_eval_duration': 869460000, 'eval_count': 432, 'eval_duration': 7533493000, 'logprobs': None, 'model_name': 'gemma4:26b', 'model_provider': 'ollama'} id='lc_run--019ea15f-fa0f-7431-a090-f1b7699a0684-0' tool_calls=[] invalid_tool_calls=[] usage_m

In [87]:
final_output = StrOutputParser().invoke(llm_output)
print("Final output:", final_output, "\n")

Final output: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals. 



In [90]:
question = "What is artificial intelligence?"
response = chain.invoke(question)
print("Response from chain:", response)

Response from chain: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals. Other definitions provided include McCarthy's description of intelligence as "the computational part of the ability to achieve goals in the world" and Marvin Minsky's description as "the ability to solve hard problems."
